# Classical Reasoning Agent (RuleTaker + CounterBench + FOLIO)

One agent, three reasoning engines, all classical:

| Data | Representation | Algorithm |
|---|---|---|
| RuleTaker | Horn rules + facts (rule-based parser) | stratified forward chaining, unification, negation as failure |
| CounterBench | Structural causal model (AND/OR/NOT equations) | abduction (backward) + intervention + forward inference |
| FOLIO | First-order logic, clause form | skolemisation + resolution refutation (unification, factoring, set of support) |

A router picks the engine from the text. No generative AI decides any answer.

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset
import time
import os
import pandas as pd

## 1. RuleTaker engines

The public `Jise/ruletaker` data uses a `Facts:` / `Rules:` format whose sentences are propositional atoms (handled by `PropReasoner`, 1b). The relational parser/reasoner below (1a) handles the classic RuleTaker templates ("If someone is young, then they are smart.") and is kept for those inputs.

In [3]:
import re, itertools
from collections import defaultdict

VARW = {'someone','something','it','they','them','he','she','everyone','everything'}

# ---------------- Parsing: natural language -> literals ----------------
def term(s):
    s = s.strip()
    if s in VARW: return '?X'
    if s.startswith('the '): s = s[4:]
    return s

def verb_norm(v):
    return v[:-1] if v.endswith('s') and not v.endswith('ss') else v

def parse_atom(t, subj=None):
    """text -> (neg, pred, args)"""
    t = t.strip()
    t = re.sub(r"\b(does|do) not\b", "not", t)
    t = t.replace("isn't", "is not").replace("aren't", "are not")
    toks = t.split()
    # elliptical conjunct: "kind" / "not kind" (inherits subject)
    if subj is not None and ' is ' not in ' '+t+' ' and ' are ' not in ' '+t+' ' and (len(toks) <= 2 or toks[0] == 'not'):
        neg = toks[0] == 'not'
        return (neg, ' '.join(toks[1:] if neg else toks), (subj,))
    m = re.match(r'^(.*?)\s+(?:is|are)\s+(not\s+)?(.+)$', t)
    if m:                                   # unary:  X is [not] attr
        return (bool(m.group(2)), m.group(3).strip(), (term(m.group(1)),))
    neg = 'not' in toks                     # binary: X verb Y
    toks = [w for w in toks if w != 'not']
    i = None
    for k in range(1, len(toks) - 1):
        if toks[k+1] == 'the' or toks[k+1] in VARW:
            i = k; break
    if i is None: i = 2 if toks[0] == 'the' and len(toks) > 3 else 1
    return (neg, verb_norm(toks[i]), (term(' '.join(toks[:i])), term(' '.join(toks[i+1:]))))

def parse_conj(text, subj=None):
    lits = []
    for p in re.split(r',\s*(?:and\s+)?|\s+and\s+', text.strip()):
        if not p.strip(): continue
        l = parse_atom(p, subj=lits[-1][2][0] if lits and len(lits[-1][2]) == 1 else subj)
        lits.append(l)
    return lits

def parse_sentence(s):
    """returns list of (body_lits, head_lit)"""
    s = s.strip().lower().rstrip('.').strip().replace(', then', ' then')
    if s.startswith('if '):
        b, h = s[3:].split(' then ', 1)
        body = parse_conj(b)
        return [(body, hl) for hl in parse_conj(h, subj='?X')]
    m = re.match(r'^(.*?)\s+(?:is|are)\s+(.*)$', s)
    subj = m.group(1) if m else ''
    if subj in ('everyone', 'everything'):
        return [([], hl) for hl in parse_conj('it is ' + m.group(2))]
    univ = re.match(r'^(?:all |every )?(.*?)\s*(?:things|people|ones)$', subj)
    if m and (subj.startswith(('all ', 'every ')) or univ):
        core = univ.group(1) if univ else subj
        core = re.sub(r'^(all|every)\s+', '', core)
        body = [(False, a.strip(), ('?X',)) for a in re.split(r',\s*(?:and\s+)?|\s+and\s+', core) if a.strip()]
        return [(body, hl) for hl in parse_conj('it is ' + m.group(2))]
    return [([], hl) for hl in parse_conj(s)]

def parse_query(q):
    q = q.strip().lower().rstrip('.?!').strip()
    return parse_conj(q)[0]

# ---------------- Reasoning: stratified forward chaining ----------------
class Reasoner:
    def __init__(self, context, mode='cwa'):
        self.mode = mode
        self.rules, self.model, self.prov = [], defaultdict(set), {}
        sents = [x for x in re.split(r'(?<=\.)\s+', context.strip()) if x.strip()]
        self.ents = set()
        for s in sents:
            for body, head in parse_sentence(s):
                self.rules.append((body, head, s.strip()))
        for body, head, _ in self.rules:
            for _, _, a in body + [(0, 0, head[2])]:
                self.ents |= {x for x in a if not x.startswith('?')}
        self._run()

    @staticmethod
    def _key(neg, pred): return ('~' if neg else '') + pred

    def _prep(self):
        prepared = []
        for body, head, txt in self.rules:
            if head[0] and self.mode == 'cwa':      # negative conclusions meaningless under CWA
                continue
            pos, naf = [], []
            for (n, p, a) in body:
                if n and self.mode == 'cwa': naf.append((p, a))
                else: pos.append((self._key(n, p), a))
            prepared.append((pos, naf, (self._key(*head[:2]), head[2]), head[1], txt))
        return prepared

    def _level(self, prepared):
        lvl = defaultdict(int)
        for _ in range(len(prepared) + 3):
            ch = False
            for pos, naf, h, hb, _ in prepared:
                need = max([lvl[p.lstrip('~')] for p, _ in pos] + [lvl[p] + 1 for p, _ in naf] + [0])
                if lvl[hb] < need: lvl[hb] = need; ch = True
            if not ch: break
        return lvl

    def _unify(self, args, tup, sub):
        sub = dict(sub)
        for a, c in zip(args, tup):
            if a.startswith('?'):
                if sub.setdefault(a, c) != c: return None
            elif a != c: return None
        return sub

    def _fire(self, rule):
        pos, naf, (hp, hargs), _, txt = rule
        out = []
        def rec(i, sub, used):
            if i == len(pos):
                free = sorted({a for _, ar in naf for a in ar if a.startswith('?') and a not in sub} |
                              {a for a in hargs if a.startswith('?') and a not in sub})
                for combo in itertools.product(self.ents, repeat=len(free)):
                    s2 = {**sub, **dict(zip(free, combo))}
                    if any(tuple(s2.get(a, a) for a in ar) in self.model[p] for p, ar in naf): continue
                    out.append(((hp, tuple(s2.get(a, a) for a in hargs)), used, [(p, tuple(s2.get(a, a) for a in ar)) for p, ar in naf]))
                return
            p, args = pos[i]
            for tup in list(self.model[p]):
                if len(tup) != len(args): continue
                s2 = self._unify(args, tup, sub)
                if s2 is not None: rec(i + 1, s2, used + [(p, tup)])
        rec(0, {}, [])
        return out

    def _run(self):
        prepared = self._prep()
        lvl = self._level(prepared)
        for L in sorted(set(lvl[r[3]] for r in prepared) | {0}):
            rs = [r for r in prepared if lvl[r[3]] == L]
            changed = True
            while changed:
                changed = False
                for r in rs:
                    for (hp, ha), used, nafs in self._fire(r):
                        if ha not in self.model[hp]:
                            self.model[hp].add(ha); changed = True
                            self.prov[(hp, ha)] = (r[4], used, nafs)

    def holds(self, lit):
        neg, pred, args = lit
        if not neg: return args in self.model[pred]
        if self.mode == 'cwa': return args not in self.model[pred]
        return args in self.model['~' + pred]

    def explain(self, atom, depth=0, seen=None):
        seen = seen if seen is not None else set()
        p, a = atom
        name = f"{p}({', '.join(a)})"
        if atom in seen or atom not in self.prov: return '  '*depth + name + '  [fact]\n'
        seen.add(atom); rule, used, nafs = self.prov[atom]
        s = '  '*depth + name + f"   <= \"{rule}\"\n"
        for u in used: s += self.explain(u, depth+1, seen)
        for p2, a2 in nafs: s += '  '*(depth+1) + f"not {p2}({', '.join(a2)})  [negation as failure]\n"
        return s

def predict(context, question, mode='cwa', cache={}):
    k = (context, mode)
    if k not in cache: cache[k] = Reasoner(context, mode)
    return cache[k].holds(parse_query(question))


### 1b. Propositional engine for the `Facts:` / `Rules:` format (linear-time forward chaining)

In [4]:

import re
from collections import defaultdict, deque

# ---------- Propositional engine for the "Facts: ... Rules: ..." RuleTaker format ----------
LEFT_CUES = r'(?:assuming|assume|if|based on|given that|given|when|whenever|since|because|as|supposing|suppose that|provided that|in case)'
RIGHT_CUES = (r'(?:then|so|thus|hence|therefore|we can (?:say|conclude|see|infer|deduce)(?: that)?|we (?:deduce|conclude|infer|know)(?: that)?|'
              r'it (?:is evident|follows|is clear|can be said|can be concluded)(?: that)?|this (?:means|implies)(?: that)?|that means(?: that)?|'
              r'(?:it is|it\'s) (?:true|certain)(?: that)?)')

def atom(t):
    t = t.lower().strip().rstrip('.!?;: ').strip()
    t = re.sub(r'\b(the|a|an)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def split_rule(line):
    """'Assuming A, we can say B.' -> ([A..], B)  (conjunction with ' and ', disjunction of bodies not supported)"""
    s = line.strip().rstrip('.').strip()
    m = re.match(rf'^{LEFT_CUES}\s+(.*?),\s*(.*)$', s, re.I)
    if m: a, b = m.group(1), m.group(2)
    else:
        m = re.match(r'^(.*?)\s+(?:implies|leads to|means that|therefore|so)\s+(.*)$', s, re.I)
        if not m: return None
        a, b = m.group(1), m.group(2)
    b = re.sub(rf'^{RIGHT_CUES}\s+', '', b.strip(), flags=re.I)
    b = re.sub(rf'^{RIGHT_CUES}\s+', '', b.strip(), flags=re.I)      # e.g. "then we can say ..."
    return a, b

class PropReasoner:
    """Forward chaining with premise counters (Dowling-Gallier, linear time) over propositional atoms."""
    def __init__(self, context):
        self.facts, self.rules, self.unparsed = [], [], []
        section = 'facts'
        for ln in [l.strip() for l in context.replace('\r', '').split('\n') if l.strip()]:
            low = ln.lower()
            if low.startswith('facts'):
                section = 'facts'; ln = ln.split(':', 1)[1].strip() if ':' in ln else ''
                if not ln: continue
            elif low.startswith('rules'):
                section = 'rules'; ln = ln.split(':', 1)[1].strip() if ':' in ln else ''
                if not ln: continue
            if section == 'facts': self.facts.append((atom(ln), ln))
            else:
                r = split_rule(ln)
                if r is None: self.unparsed.append(ln); continue
                body = [atom(x) for x in re.split(r'\s+and\s+', r[0]) if x.strip()]
                heads = [atom(x) for x in re.split(r'\s+and\s+', r[1]) if x.strip()]
                for h in heads: self.rules.append((body, h, ln))
        self._run()

    def _run(self):
        self.derived, self.prov = {}, {}
        watch = defaultdict(list); count = []
        for i, (body, h, txt) in enumerate(self.rules):
            body_set = set(body); count.append(len(body_set))
            for b in body_set: watch[b].append(i)
        q = deque()
        for a, ln in self.facts:
            if a not in self.derived: self.derived[a] = True; self.prov[a] = ('fact', []); q.append(a)
        for i, (body, h, txt) in enumerate(self.rules):
            if count[i] == 0 and h not in self.derived:
                self.derived[h] = True; self.prov[h] = (txt, []); q.append(h)
        while q:
            a = q.popleft()
            for i in watch[a]:
                count[i] -= 1
                if count[i] == 0:
                    h = self.rules[i][1]
                    if h not in self.derived:
                        self.derived[h] = True; self.prov[h] = (self.rules[i][2], self.rules[i][0]); q.append(h)

    def holds(self, statement):
        a = atom(statement)
        if a in self.derived: return True
        m = re.match(r'^(.*?)\s+(?:is|are|does|do|did)?\s*not\s+(.*)$', a)      # closed world: negation as failure
        if m:
            pos = atom(m.group(1) + ' ' + m.group(2))
            return pos not in self.derived
        return False

    def explain(self, statement, depth=0):
        a = atom(statement)
        if a not in self.prov: return '  ' * depth + a + '  [not derived]\n'
        rule, prem = self.prov[a]
        s = '  ' * depth + a + (f'   <= "{rule}"' if rule != 'fact' else '  [fact]') + '\n'
        for p in prem: s += self.explain(p, depth + 1)
        return s

_prop_cache = {}
def prop_predict(context, statement):
    if context not in _prop_cache: _prop_cache[context] = PropReasoner(context)
    return _prop_cache[context].holds(statement)


## 2. CounterBench engine (counterfactual reasoning over a structural causal model)

In [5]:
import re

# ---------------- CounterBench: text -> structural causal model ----------------
def _lit(item):
    item = item.strip().lower()
    neg = False
    while item.startswith('not '):
        neg = not neg; item = item[4:].strip()
    item = re.sub(r'^(an? individual is|the individual is|an individual|individual is)\s+', '', item).strip()
    return item, neg

def _split_items(s):
    return [x.strip() for x in re.split(r',\s*(?:and\s+)?|\s+and\s+', s) if x.strip()]

def parse_scm(info):
    """returns (eqs, observed) ; eqs[var] = (op, [(parent, neg), ...])"""
    t = re.sub(r'\s+', ' ', info.strip())
    low = t.lower()
    m = re.search(r'we know that (.*?)(?=\s*(?:we observed|would |[a-z]+ bern\(|[a-z]+ is unobserved)|$)', low)
    seg = m.group(1) if m else low
    seg = re.sub(r'\b\w+ bern\([0-9.]+\)', '', seg)
    seg = re.sub(r'\.\s+', ', ', seg)
    pieces = [p.strip() for p in re.split(r',\s*', seg.rstrip('. ')) if p.strip()]
    pieces = [re.sub(r'^and\s+', '', p) for p in pieces]
    clauses = []                                      # [lhs, [rhs items]]
    for p in pieces:
        mm = re.match(r'^(.*?)\s+(?:together\s+)?causes?\s+(.*)$', p)
        if mm:
            clauses.append([mm.group(1), _split_items(mm.group(2))])
        elif clauses:                                 # continuation of an RHS list
            clauses[-1][1] += _split_items(p)
    eqs = {}
    for lhs, rhs in clauses:
      if ' or ' in lhs:
        op, parts = 'or', [x for x in re.split(r'\s+or\s+', lhs)]
      else:
        op, parts = 'and', [x for x in re.split(r'\s+and\s+', lhs)]

      ins = [_lit(x) for x in parts]
      for r in rhs:
        v, vneg = _lit(r)

        # Apply negative effect using De Morgan's law
        effective_op = op
        effective_ins = ins

        if vneg:
            effective_op = 'or' if op == 'and' else 'and'
            effective_ins = [
                (parent, not neg)
                for parent, neg in ins
            ]

        eqs[v] = (effective_op, effective_ins)
    obs = {}
    for mo in re.finditer(r'we observed ([^.]*)\.', low):
        for it in _split_items(mo.group(1)):
            v, n = _lit(it); obs[v] = 0 if n else 1
    return eqs, obs

def parse_question(q):
    low = re.sub(r'\s+', ' ', q.strip().lower()).rstrip('?. ')
    ym = re.match(r'^(?:.*?\.\s*)?would (?:an individual is |the individual is |an individual )?(.*?) (?:occur|happen|be true)?\s*(?:if (.*))?$', low)
    y_text, cond = ym.group(1), ym.group(2)
    y, yneg = _lit(y_text.replace(' occur', ''))
    ivs, factual = {}, {}
    if 'assume' in low or 'suppose' in low:
        for mm in re.finditer(r'(?:assume|suppose)\s+(not\s+)?([a-z0-9_]+)', low):
            ivs[mm.group(2)] = 0 if mm.group(1) else 1
    elif cond:
        main, _, inst = cond.partition(' instead of ')
        for it in _split_items(main):
            v, n = _lit(it); ivs[v] = 0 if n else 1
        if inst:
            v, n = _lit(inst)
            factual[v] = 0 if n else 1
    return y, yneg, ivs, factual

def solve_counterfactual(info, question, trace=None, root_default=0, use_factual=True):
    eqs, obs = parse_scm(info)
    y, yneg, ivs, factual = parse_question(question)
    # --- abduction: propagate "must hold" values backwards
    req, stack = {}, list(obs.items()) + (list(factual.items()) if use_factual else [])
    while stack:
        v, val = stack.pop()
        if v in req: continue
        req[v] = val
        if v not in eqs: continue
        op, ins = eqs[v]
        if (op == 'and' and val == 1) or (op == 'or' and val == 0) or len(ins) == 1:
            for p, neg in ins:
                stack.append((p, (val ^ neg) if len(ins) == 1 else (1 ^ neg if op == 'and' else 0 ^ neg)))
    # --- forward inference in the intervened model M_x
    memo = {}
    def val(v):
        if v in ivs: return ivs[v]
        if v in memo: return memo[v]
        memo[v] = 0                                      # cycle guard
        if v not in eqs:
            r = req.get(v, root_default)
        else:
            op, ins = eqs[v]
            vs = [val(p) ^ int(n) for p, n in ins]
            r = int(all(vs)) if op == 'and' else int(any(vs))
        memo[v] = r
        if trace is not None: trace.append((v, r))
        return r
    out = val(y) ^ int(yneg)
    return bool(out)


## 3. FOLIO engine (first-order logic parser + resolution prover)

In [6]:
import re, time, itertools

# ================= FOL parsing (FOLIO syntax) =================
TOK = re.compile(r'\s*(∀|∃|¬|∧|∨|⊕|→|↔|⟶|⇒|⇔|≠|=|\(|\)|,|[A-Za-z_][A-Za-z0-9_\-\']*|-?\d+)')
def tokenize(s):
    s = s.replace('\\forall','∀').replace('\\exists','∃').replace('\\neg','¬').replace('\\land','∧').replace('\\lor','∨')
    s = s.replace('.', ' ').replace('[','(').replace(']',')')
    pos, out = 0, []
    s = s.strip()
    while pos < len(s):
        m = TOK.match(s, pos)
        if not m: pos += 1; continue
        out.append(m.group(1)); pos = m.end()
    return out

class Parser:
    def __init__(self, toks): self.t, self.i = toks, 0
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        v = self.peek()
        if x and v != x: raise ValueError(f'expected {x} got {v}')
        self.i += 1; return v
    def iff(self, bound):
        l = self.imp(bound)
        while self.peek() in ('↔', '⇔'): self.eat(); l = ('iff', l, self.imp(bound))
        return l
    def imp(self, bound):
        l = self.orx(bound)
        if self.peek() in ('→', '⟶', '⇒'): self.eat(); return ('imp', l, self.imp(bound))
        return l
    def orx(self, bound):
        l = self.andx(bound)
        while self.peek() in ('∨', '⊕'):
            op = self.eat(); r = self.andx(bound); l = ('or' if op == '∨' else 'xor', l, r)
        return l
    def andx(self, bound):
        l = self.unary(bound)
        while self.peek() == '∧': self.eat(); l = ('and', l, self.unary(bound))
        return l
    def term(self, bound):
        name = self.eat()
        if self.peek() == '(':
            self.eat('('); args = [self.term(bound)]
            while self.peek() == ',': self.eat(); args.append(self.term(bound))
            self.eat(')'); return (name.lower(), *args)
        return (('?' + name),) if name in bound else (name.lower(),)
    def unary(self, bound):
        v = self.peek()
        if v == '¬': self.eat(); return ('not', self.unary(bound))
        if v in ('∀', '∃'):
            self.eat(); var = self.eat(); b2 = bound | {var}
            body = self.unary(b2) if self.peek() in ('(', '¬', '∀', '∃') else self.iff(b2)
            return ('all' if v == '∀' else 'ex', var, body)
        if v == '(':
            self.eat('('); f = self.iff(bound); self.eat(')'); return self.maybe_eq(f)
        # atom or equality
        name = self.eat()
        if self.peek() == '(':
            self.eat('('); args = [self.term(bound)]
            while self.peek() == ',': self.eat(); args.append(self.term(bound))
            self.eat(')'); return ('atom', name.lower(), tuple(args))
        self.i -= 1
        t1 = self.term(bound)
        if self.peek() in ('=', '≠'):
            op = self.eat(); t2 = self.term(bound); a = ('atom', 'eq', (t1, t2))
            return a if op == '=' else ('not', a)
        return ('atom', name.lower(), ())
    def maybe_eq(self, f): return f

def parse_fol(s):
    p = Parser(tokenize(s)); f = p.iff(frozenset())
    return f

# ================= Clausification =================
_cnt = itertools.count()
def elim(f):
    k = f[0]
    if k == 'atom': return f
    if k == 'not': return ('not', elim(f[1]))
    if k in ('and', 'or'): return (k, elim(f[1]), elim(f[2]))
    if k == 'imp': return ('or', ('not', elim(f[1])), elim(f[2]))
    if k == 'iff':
        a, b = elim(f[1]), elim(f[2]); return ('and', ('or', ('not', a), b), ('or', ('not', b), a))
    if k == 'xor':
        a, b = elim(f[1]), elim(f[2]); return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    return (k, f[1], elim(f[2]))

def nnf(f, neg=False):
    k = f[0]
    if k == 'atom': return ('not', f) if neg else f
    if k == 'not': return nnf(f[1], not neg)
    if k in ('and', 'or'):
        k2 = ({'and': 'or', 'or': 'and'}[k]) if neg else k
        return (k2, nnf(f[1], neg), nnf(f[2], neg))
    k2 = ({'all': 'ex', 'ex': 'all'}[k]) if neg else k
    return (k2, f[1], nnf(f[2], neg))

def subst_term(t, m):
    if t[0].startswith('?'): return m.get(t[0], t)
    return (t[0], *[subst_term(a, m) for a in t[1:]])

def skolemize(f, m=None, univ=()):
    m = m or {}; k = f[0]
    if k == 'atom': return ('atom', f[1], tuple(subst_term(a, m) for a in f[2]))
    if k == 'not': return ('not', skolemize(f[1], m, univ))
    if k in ('and', 'or'): return (k, skolemize(f[1], m, univ), skolemize(f[2], m, univ))
    v = '?' + f[1]
    if k == 'all':
        nv = f'?{f[1]}_{next(_cnt)}'
        return skolemize(f[2], {**m, v: (nv,)}, univ + (nv,))
    sk = f'sk{next(_cnt)}'
    return skolemize(f[2], {**m, v: (sk, *[(u,) for u in univ])}, univ)

def to_clauses(f):
    f = skolemize(nnf(elim(f)))
    def cnf(g):
        if g[0] == 'and': return cnf(g[1]) + cnf(g[2])
        if g[0] == 'or':
            a, b = cnf(g[1]), cnf(g[2])
            if len(a) * len(b) > 4000: raise ValueError('CNF blowup')
            return [x | y for x in a for y in b]
        if g[0] == 'not': return [frozenset([(False, g[1][1], g[1][2])])]
        return [frozenset([(True, g[1], g[2])])]
    out = []
    for c in cnf(f):
        if any((not s, p, a) in c for s, p, a in c): continue     # tautology
        out.append(c)
    return out

# ================= Unification + resolution =================
def walk(t, s):
    while t[0].startswith('?') and t in s: t = s[t]
    return t
def occurs(v, t, s):
    t = walk(t, s)
    return t == v or (not t[0].startswith('?') and any(occurs(v, a, s) for a in t[1:]))
def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if a[0].startswith('?'):
        if occurs(a, b, s): return None
        return {**s, a: b}
    if b[0].startswith('?'): return unify(b, a, s)
    if a[0] != b[0] or len(a) != len(b): return None
    for x, y in zip(a[1:], b[1:]):
        s = unify(x, y, s)
        if s is None: return None
    return s
def apply(t, s):
    t = walk(t, s)
    return t if t[0].startswith('?') else (t[0], *[apply(a, s) for a in t[1:]])
def rename(c):
    n = next(_cnt); m = {}
    def r(t):
        if t[0].startswith('?'): return m.setdefault(t, (f'?r{n}_{len(m)}',))
        return (t[0], *[r(a) for a in t[1:]])
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in c)
def apply_lit(l, s): return (l[0], l[1], tuple(apply(a, s) for a in l[2]))
def tsize(c): return sum(1 + sum(len(str(a)) for a in args) for _, _, args in c)

def factors(c):
    out = [c]; lits = list(c)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = {}
                for x, y in zip(a[2], b[2]):
                    s = unify(x, y, s) if s is not None else None
                if s is not None: out.append(frozenset(apply_lit(l, s) for l in c))
    return out

def resolve_all(c1, c2):
    c2 = rename(c2); res = []
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                s = {}
                for x, y in zip(l1[2], l2[2]):
                    s = unify(x, y, s) if s is not None else None
                if s is None: continue
                r = frozenset(apply_lit(l, s) for l in (c1 - {l1}) | (c2 - {l2}))
                if any((not sg, p, a) in r for sg, p, a in r): continue
                res.append(r)
    return res

def canon(c):
    m = {}
    def r(t):
        if t[0].startswith('?'): return m.setdefault(t, ('?v%d' % len(m),))
        return (t[0], *[r(a) for a in t[1:]])
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in sorted(c, key=str))

def refute(usable, sos, time_limit=3.0, max_clauses=20000):
    """set-of-support binary resolution with factoring; True iff empty clause derived"""
    t0 = time.time()
    seen = {canon(c) for c in usable} | {canon(c) for c in sos}
    usable, sos = list(usable), sorted(sos, key=lambda c: (len(c), tsize(c)))
    if any(len(c) == 0 for c in sos + usable): return True
    steps = 0
    while sos:
        if time.time() - t0 > time_limit or len(seen) > max_clauses: return None
        sos.sort(key=lambda c: (len(c), tsize(c)))
        g = sos.pop(0)
        for other in usable + [g]:
            for r in resolve_all(g, other):
                for f in factors(r):
                    if len(f) == 0: return True
                    if len(f) > 8: continue
                    k = canon(f)
                    if k not in seen: seen.add(k); sos.append(f)
        usable.append(g); steps += 1
    return False                                   # saturated: not provable

class FolKB:
    def __init__(self, premises_fol):
        self.cl = []
        self.has_eq = False
        for p in premises_fol:
            f = parse_fol(p)
            self.cl += to_clauses(f)
        if any(l[1] == 'eq' for c in self.cl for l in c):
            x = ('?x',); self.cl.append(frozenset([(True, 'eq', (x, x))]))
    def entails(self, concl_fol, time_limit=3.0):
        neg = to_clauses(('not', parse_fol(concl_fol)))
        return refute(self.cl, neg, time_limit)
    def label3(self, concl_fol, time_limit=3.0):
        if self.entails(concl_fol, time_limit): return 'True'
        pos = to_clauses(parse_fol(concl_fol))
        if refute(self.cl, pos, time_limit): return 'False'
        return 'Uncertain'


## 4. Heuristic English → FOL translator (used when no FOL annotation is given)

In [7]:
import re
ART = {'a', 'an', 'the'}
AUX = r"(?:is not|are not|isn't|aren't|is|are|was|were|does not|do not|doesn't|don't|cannot|can not|can|will not|will|must|has|have|had)"

def _stem(w):
    if len(w) > 3 and w.endswith('ies'): return w[:-3] + 'y'
    if len(w) > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')): return w[:-1]
    return w

def norm_pred(ph):
    toks = [t for t in re.findall(r"[a-z0-9']+", ph.lower()) if t not in ART and t not in ('is', 'are', 'be', 'being', 'was', 'were', 'to')]
    return '_'.join(_stem(t) for t in toks)

def pred_expr(phrase, v):
    ph = phrase.strip(' ,')
    neg = False
    m = re.match(r'^(not|no longer|never)\s+(.*)$', ph)
    if m: neg, ph = True, m.group(2)
    for sep, op in ((' or ', '∨'), (' and ', '∧')):
        if sep in ph:
            parts = [pred_expr(p, v) for p in ph.split(sep)]
            if None in parts: return None
            e = '(' + f' {op} '.join(parts) + ')'
            return f'¬{e}' if neg else e
    p = norm_pred(ph)
    if not p: return None
    return f'¬{p}({v})' if neg else f'{p}({v})'

def _const(sub, last):
    sub = sub.strip().lower()
    if sub in ('he', 'she', 'it', 'they', 'him', 'her', 'them'): return last[0] if last else None
    c = '_'.join(w for w in re.findall(r"[a-z0-9']+", sub) if w not in ART)
    return c or None

def _split_aux(s):
    m = re.match(rf"^(.*?)\s+({AUX})\s+(.*)$", s)
    if m: return m.group(1), m.group(2), m.group(3)
    toks = s.split()
    if len(toks) >= 2: return toks[0], '', ' '.join(toks[1:])        # "Every dog barks"
    return None

def _restr(np, v):
    m = re.match(r'^(.*?)\s+(?:who|that|which)\s+(.*)$', np)
    if m:
        head = pred_expr(m.group(1), v)
        rel = m.group(2)
        mm = re.match(rf'^({AUX})\s+(.*)$', rel)
        relp = pred_expr(('not ' if mm and 'not' in mm.group(1) else '') + (mm.group(2) if mm else rel), v)
        return f'({head} ∧ {relp})' if head and relp else None
    return pred_expr(np, v)

def _atomic(s, last):
    sp = _split_aux(s)
    if not sp: return None
    sub, aux, pred = sp
    neg = ('not' in aux) or aux in ("isn't", "aren't", "doesn't", "don't", "cannot")
    c = _const(sub, last)
    if not c or len(c.split('_')) > 3: return None
    last[:] = [c]
    verbpart = pred if aux in ('', 'is', 'are', 'was', 'were', 'is not', 'are not') else f'{aux.split()[0]} {pred}'
    p = pred_expr(verbpart if aux in ('', 'is', 'are', 'was', 'were', 'is not', 'are not', "isn't", "aren't") else pred, c)
    if p is None: return None
    return f'¬{p}' if neg and not p.startswith('¬') else p

def translate(sent, last=None):
    last = last if last is not None else []
    s = sent.strip().rstrip('.').strip(); low = s.lower()
    low = re.sub(r'^either\s+', '', low)
    m = re.match(r'^if (.*?),? then (.*)$', low) or re.match(r'^if (.*?), (.*)$', low)
    if m:
        a, b = translate(m.group(1), last), translate(m.group(2), last)
        return f'(({a}) → ({b}))' if a and b else None
    if ' if and only if ' in low:
        a, b = low.split(' if and only if ', 1); a, b = translate(a, last), translate(b, last)
        return f'(({a}) ↔ ({b}))' if a and b else None
    for sep, op in ((', but ', '∧'), (', and ', '∧'), (' but ', '∧'), (', or ', '∨'), (' or ', '∨'), (' and ', '∧')):
        if sep in low and not re.match(r'^(all|every|each|any|no|some)\b', low):
            parts = low.split(sep, 1)
            l = translate(parts[0], last); r = translate(parts[1], last)
            if l and r: return f'(({l}) {op} ({r}))'
    m = re.match(r'^(all|every|each|any)\s+(.*)$', low)
    if m:
        sp = _split_aux(m.group(2))
        if not sp: return None
        np, aux, pred = sp
        r = _restr(np, 'x'); neg = 'not' in aux or aux in ("isn't", "aren't", "doesn't", "don't", "cannot")
        c = pred_expr(('not ' if neg else '') + pred, 'x')
        return f'∀x (({r}) → ({c}))' if r and c else None
    m = re.match(r'^no\s+(.*)$', low)
    if m:
        sp = _split_aux(m.group(1))
        if not sp: return None
        r = _restr(sp[0], 'x'); c = pred_expr(sp[2], 'x')
        return f'∀x (({r}) → ¬({c}))' if r and c else None
    m = re.match(r'^(?:some|at least one|there (?:is|are|exists?) (?:some|an?|at least one))\s+(.*)$', low)
    if m:
        body = m.group(1)
        sp = _split_aux(body)
        if not sp: return None
        r = _restr(sp[0], 'x'); c = pred_expr(sp[2], 'x') if sp[1] or len(sp[0].split()) else None
        return f'∃x (({r}) ∧ ({c}))' if r and c else None
    return _atomic(low, last)

def nl_to_fol(premises, conclusion):
    """premises: list/str of sentences. returns (premise_fols, conclusion_fol or None)"""
    if isinstance(premises, str):
        premises = [p for p in re.split(r'\n|(?<=\.)\s+', premises) if p.strip()]
    last, out = [], []
    for p in premises:
        try: f = translate(p, last)
        except Exception: f = None
        if f: out.append(f)
    try: c = translate(conclusion, last)
    except Exception: c = None
    return out, c


## 5. Router and unified `solve()`

In [8]:
import re, time
from collections import defaultdict

def _looks_like_fol(text):
    """Detect explicit first-order logic notation, not ordinary English."""
    if not isinstance(text, str):
        return False
    markers = ("∀", "∃", "→", "↔", "∧", "∨", "¬", "\\forall", "\\exists", "\\land", "\\lor")
    return any(m in text for m in markers)

def detect_task(context, statement):
    """Choose an engine from the structure of the supplied knowledge base."""
    c = str(context).lower()
    s = str(statement).lower()

    # Explicit dataset schemas are handled by solve_record() at evaluation time.
    if "facts:" in c and "rules:" in c:
        return "ruletaker"
    if "we know that" in c and (" causes " in c or " cause " in c) and (
        s.strip().startswith(("would ", "assume ", "suppose "))
    ):
        return "counterbench"
    if _looks_like_fol(str(context)) or _looks_like_fol(str(statement)):
        return "folio"

    sents = [x for x in re.split(r"(?<=\.)\s+|\n", str(context).strip()) if x.strip()]
    if sents:
        short = sum(
            1 for x in sents
            if len(x.split()) <= 9 or (
                x.lower().startswith("if ") and " then " in x.lower() and len(x.split()) <= 16
            )
        )
        if short / len(sents) >= 0.8:
            return "ruletaker"
    return "folio"

_fol_cache = {}

def folio_predict(premises, conclusion, premises_fol=None, conclusion_fol=None, time_limit=3.0):
    """Return True only when classical resolution proves entailment."""
    try:
        if premises_fol is None:
            pf, cf = nl_to_fol(premises, conclusion)
        else:
            pf = premises_fol
            cf = conclusion_fol
        if not cf or not pf:
            return False
        key = tuple(pf)
        if key not in _fol_cache:
            _fol_cache[key] = FolKB(pf)
        result = _fol_cache[key].entails(cf, time_limit)
        return result is True
    except Exception as exc:
        # Keep the binary interface, but expose the failure in diagnostics.
        _FOL_ERRORS.append(str(exc))
        return False

_FOL_ERRORS = []

def solve(context, statement, task=None, mode="cwa",
          premises_fol=None, conclusion_fol=None):
    """Unified solver. The final True/False decision is made by a classical engine."""
    context = str(context)
    statement = str(statement)
    task = task or detect_task(context, statement)

    if task == "counterbench":
        return bool(solve_counterfactual(context, statement))
    if task == "folio":
        if premises_fol is None and (_looks_like_fol(context) or _looks_like_fol(statement)):
            # Evaluation may provide the logical representation directly.
            direct_premises = [
                line.strip() for line in context.splitlines()
                if line.strip() and not line.strip().lower().startswith(("premises:", "knowledge base:"))
            ]
            return bool(folio_predict(
                "", "",
                premises_fol=direct_premises,
                conclusion_fol=statement
            ))
        return bool(folio_predict(
            context, statement,
            premises_fol=premises_fol,
            conclusion_fol=conclusion_fol
        ))
    if "facts:" in context.lower() and "rules:" in context.lower():
        return bool(prop_predict(context, statement))
    return bool(predict(context, statement, mode))

## 5b. Evaluation adapter: accept the logic supplied at evaluation time

The private evaluation file may use different column names for the same logical inputs.
This adapter recognizes the assignment's common schemas and uses explicit FOL columns
when they are supplied. It does not generate labels from a language model.


In [9]:
def _first_present(row, names, required=True):
    for name in names:
        if name in row and pd.notna(row[name]):
            return row[name]
    if required:
        raise KeyError(f"None of the expected columns are present: {names}")
    return None

def solve_record(row, mode="cwa"):
    """
    Predict one row from a supplied evaluation table.

    Supported schemas:
      RuleTaker: context + statement
      CounterBench: given_info + question
      FOLIO: premises + conclusion, optionally premises-FOL + conclusion-FOL
      Direct FOL: premises_fol + conclusion_fol (or premises-FOL + conclusion-FOL)
    """
    keys = set(row.index)

    # CounterBench has a distinctive input schema.
    if {"given_info", "question"}.issubset(keys):
        return bool(solve(
            row["given_info"], row["question"], task="counterbench", mode=mode
        ))

    # FOLIO with gold logical forms: use the supplied logic directly.
    pfol = _first_present(
        row, ["premises-FOL", "premises_fol", "premises_logic", "logic"],
        required=False
    )
    cfol = _first_present(
        row, ["conclusion-FOL", "conclusion_fol", "conclusion_logic", "query_logic"],
        required=False
    )
    if pfol is not None and cfol is not None:
        if isinstance(pfol, str):
            pfol = [x.strip() for x in str(pfol).splitlines() if x.strip()]
        return bool(folio_predict(
            premises="", conclusion="",
            premises_fol=list(pfol), conclusion_fol=str(cfol),
            time_limit=5.0
        ))

    # FOLIO in natural language (the translator is a heuristic, so this is less reliable).
    if {"premises", "conclusion"}.issubset(keys):
        return bool(solve(
            row["premises"], row["conclusion"], task="folio", mode=mode
        ))

    # RuleTaker / generic context-query format.
    context = _first_present(row, ["context", "knowledge_base", "kb"])
    statement = _first_present(row, ["statement", "query", "question"])
    return bool(solve(context, statement, mode=mode))

def infer_input_columns(df):
    """Return the detected input format and relevant column names."""
    cols = set(df.columns)
    if {"given_info", "question"}.issubset(cols):
        return "CounterBench", ["given_info", "question"]
    if ({"premises-FOL", "conclusion-FOL"}.issubset(cols) or
        {"premises_fol", "conclusion_fol"}.issubset(cols) or
        {"logic", "query_logic"}.issubset(cols)):
        return "FOLIO / supplied FOL", ["premises-FOL or premises_fol", "conclusion-FOL or conclusion_fol"]
    if {"premises", "conclusion"}.issubset(cols):
        return "FOLIO natural language", ["premises", "conclusion"]
    if ({"context", "statement"}.issubset(cols) or
        {"context", "query"}.issubset(cols) or
        {"context", "question"}.issubset(cols)):
        return "RuleTaker / context-query", ["context", "statement/query/question"]
    if {"knowledge_base", "query"}.issubset(cols) or {"kb", "query"}.issubset(cols):
        return "Generic knowledge-base/query", ["knowledge_base or kb", "query"]
    return "Unrecognized", []

## 6. Sanity checks and reasoning traces

In [10]:
# RuleTaker
ctx = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful."
print(solve(ctx, "Bob is successful."))                       # True
print(Reasoner(ctx).explain(('successful', ('bob',))))
# RuleTaker, Facts/Rules format (Jise/ruletaker)
pc = "Facts:\nCharlie is fast.\nRules:\nBased on charlie is fast, it is evident that the house volunteers.\nGiven that the house volunteers, it follows that charlie is a chef."
print(solve(pc, "Charlie is a chef."), solve(pc, "The river volunteers."))   # True False
print(PropReasoner(pc).explain("Charlie is a chef."))
# CounterBench
info = "We know that Nuv causes Splee, Blen and Druk, not Druk causes Plog, Plog causes Skrim, Skrim or Druk causes Zimb, Zimb causes Yurd, and Yurd causes Wrox."
print(solve(info, "Would Wrox occur if not Nuv and not Splee?"))
# FOLIO with gold-style FOL
kb = FolKB(["∀x (Human(x) → Mortal(x))", "Human(socrates)"])
print(kb.label3("Mortal(socrates)"), kb.label3("¬Mortal(socrates)"), kb.label3("Wise(socrates)"))
# FOLIO from raw English (heuristic translator)
print(solve("All students who study hard are successful. Tom is a student. Tom studies hard.", "Tom is successful.", task='folio'))

True
successful(bob)   <= "If someone is smart and kind, then they are successful."
  smart(bob)   <= "If someone is young, then they are smart."
    young(bob)   <= "Bob is young."
  kind(bob)   <= "Bob is kind."

True False
charlie is chef   <= "Given that the house volunteers, it follows that charlie is a chef."
  house volunteers   <= "Based on charlie is fast, it is evident that the house volunteers."
    charlie is fast  [fact]

True
True False Uncertain
True


## 7. Public-dataset experiments

FOLIO is a **gated** dataset. First accept the dataset access conditions on its Hugging Face page. Then run the Hugging Face login cell below before loading FOLIO.

In [11]:
# ================= HUGGING FACE LOGIN =================
# Run this cell before the FOLIO dataset-loading cell.

from huggingface_hub import login, whoami

print("Hugging Face login")
login()  # A secure token-entry prompt will appear.

# Verify that authentication succeeded.
user_info = whoami()
print("Successfully logged in as:", user_info["name"])

Hugging Face login
Successfully logged in as: tech07mohit


In [12]:
def to_bool(y):
    if isinstance(y, bool):
        return y

    if isinstance(y, (int, float)):
        return bool(y)

    if isinstance(y, str):
        return y.strip().lower() in ["true", "1", "yes", "entailed"]

    raise ValueError(f"Unknown label: {y}")

In [13]:
def report(name, correct, total, elapsed, extra=""):
    accuracy = (correct / total * 100) if total > 0 else 0

    print("\nDataset:", name)
    print("Correct Predictions:", correct, "/", total)
    print("Accuracy:", round(accuracy, 2), "%")
    print("Execution Time:", round(elapsed, 2), "seconds")
    print(extra)

In [14]:
rt = load_dataset("Jise/ruletaker")

for split in rt:
    df = rt[split].to_pandas()

    t0 = time.time()
    c = 0
    unparsed = 0
    seen = set()

    for ctx_, st, y in zip(df['context'], df['statement'], df['flag']):
        c += (solve(ctx_, st) == to_bool(y))

        if ctx_ not in seen:
            seen.add(ctx_)
            unparsed += len(_prop_cache[ctx_].unparsed) if ctx_ in _prop_cache else 0

    report(
        f"RuleTaker/{split}",
        c,
        len(df),
        time.time() - t0,
        f"(unparsed rule lines: {unparsed})"
    )


Dataset: RuleTaker/train
Correct Predictions: 983 / 1000
Accuracy: 98.3 %
Execution Time: 0.2 seconds
(unparsed rule lines: 0)

Dataset: RuleTaker/test
Correct Predictions: 244 / 250
Accuracy: 97.6 %
Execution Time: 0.05 seconds
(unparsed rule lines: 0)

Dataset: RuleTaker/ood_test
Correct Predictions: 373 / 375
Accuracy: 99.47 %
Execution Time: 0.07 seconds
(unparsed rule lines: 0)


In [16]:
import pandas as pd
import time
from collections import defaultdict
from datasets import load_dataset

def load_counterbench():
    try:
        d = load_dataset("CounterBench/CounterBench")
        return pd.concat(
            [d[s].to_pandas() for s in d]
        )
    except Exception:
        from huggingface_hub import hf_hub_download

        frames = []

        for fn in (
            "data_balanced_alpha_V1.json",
            "data_balanced_backdoor_V2.json"
        ):
            p = hf_hub_download(
                "CounterBench/CounterBench",
                fn,
                repo_type="dataset"
            )

            try:
                frames.append(pd.read_json(p))
            except ValueError:
                frames.append(pd.read_json(p, lines=True))

        return pd.concat(frames, ignore_index=True)


cb = load_counterbench()

print("CounterBench loaded successfully")
print("Total examples:", len(cb))
print("Columns:", cb.columns.tolist())

Generating train split: 0 examples [00:00, ? examples/s]

CounterBench loaded successfully
Total examples: 1200
Columns: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']


In [17]:

# ---------- CounterBench Evaluation ----------
cb = load_counterbench()

print("CounterBench dataset size:", len(cb))

for kw in ({}, {'use_factual': False}, {'root_default': 1}):

    t0 = time.time()
    correct = 0
    processed = 0
    failures = []

    per_type = defaultdict(lambda: [0, 0])

    for idx, (info, q, a, ty) in enumerate(
        zip(cb['given_info'], cb['question'],
            cb['answer'], cb['type'])
    ):
        try:
            prediction = solve_counterfactual(info, q, **kw)
            actual = to_bool(a)

            processed += 1
            is_correct = (prediction == actual)

            correct += is_correct
            per_type[ty][0] += is_correct
            per_type[ty][1] += 1

        except Exception as e:
            failures.append((idx, str(e)))

    elapsed = time.time() - t0

    print("\nCounterBench:", kw)
    print("Total examples:", len(cb))
    print("Successfully processed:", processed)
    print("Processing errors:", len(failures))

    if processed:
        print(
            "Accuracy:",
            round(correct / processed * 100, 2), "%"
        )

    print("Inference time:", round(elapsed, 3), "seconds")

    print("Accuracy by question type:")
    for name, values in per_type.items():
        print(
            name,
            round(values[0] / values[1] * 100, 2), "%"
        )

    if failures:
        print("First 5 errors:", failures[:5])


Generating train split: 0 examples [00:00, ? examples/s]

CounterBench dataset size: 1200

CounterBench: {}
Total examples: 1200
Successfully processed: 1200
Processing errors: 0
Accuracy: 94.58 %
Inference time: 0.234 seconds
Accuracy by question type:
basic 100.0 %
conditional 90.0 %
joint 96.0 %
nested 96.0 %
backdoor 90.0 %

CounterBench: {'use_factual': False}
Total examples: 1200
Successfully processed: 1200
Processing errors: 0
Accuracy: 91.67 %
Inference time: 0.357 seconds
Accuracy by question type:
basic 100.0 %
conditional 90.0 %
joint 96.0 %
nested 96.0 %
backdoor 72.5 %

CounterBench: {'root_default': 1}
Total examples: 1200
Successfully processed: 1200
Processing errors: 0
Accuracy: 95.0 %
Inference time: 0.152 seconds
Accuracy by question type:
basic 100.0 %
conditional 92.0 %
joint 96.0 %
nested 96.0 %
backdoor 90.0 %


In [18]:

# ---------- CounterBench: Detailed causal structure analysis ----------

from collections import Counter

duplicate_effects = Counter()
negative_effects = 0
joint_causes = 0
multiple_mechanisms = []

for info in cb['given_info']:

    eqs, obs = parse_scm(info)

    # Count repeated effects
    for variable in eqs:
        duplicate_effects[variable] += 1

    # Count negative causal effects
    if re.search(r'causes?\s+not\s+', info, re.I):
        negative_effects += 1

    # Count joint causal mechanisms
    if re.search(r'\b\w+\s+and\s+\w+\s+together\s+cause', info, re.I):
        joint_causes += 1

    # Find repeated RHS variables in the original context
    rhs = re.findall(
        r'\bcauses?\s+(?:not\s+)?([a-z0-9_]+)',
        info.lower()
    )

    counts = Counter(rhs)

    for variable, count in counts.items():
        if count > 1:
            multiple_mechanisms.append((info, variable, count))

print("Negative causal examples:", negative_effects)
print("Joint causal examples:", joint_causes)
print("Repeated effects:", len(multiple_mechanisms))

print("\nExamples of repeated effects:")

for info, variable, count in multiple_mechanisms[:10]:
    print("\nContext:", info)
    print("Repeated variable:", variable)
    print("Occurrences:", count)

print("\nQuestion types:")
print(cb['type'].value_counts())


Negative causal examples: 580
Joint causal examples: 680
Repeated effects: 0

Examples of repeated effects:

Question types:
type
basic          250
conditional    250
joint          250
nested         250
backdoor       200
Name: count, dtype: int64


In [19]:
# ---------- FOLIO (yale-nlp/FOLIO): premises, premises-FOL, conclusion, conclusion-FOL, label ----------
try:
    fo = load_dataset("yale-nlp/FOLIO")
except Exception as e:
    print("FOLIO not loaded (gated dataset: accept its terms on the dataset page and run login() first):", e)
    fo = {}
for split in fo:
    df = fo[split].to_pandas().head(300)
    for use_gold in (True, False):
        t0 = time.time(); c3 = c2 = 0; _fol_cache.clear()
        for _, r in df.iterrows():
            if use_gold:
                pf = [x for x in str(r['premises-FOL']).split('\n') if x.strip()]
                cf = r['conclusion-FOL']
                try: lab = FolKB(pf).label3(cf, 2.0)
                except Exception: lab = 'Uncertain'
                p = (lab == 'True')
            else:
                p = folio_predict(r['premises'], r['conclusion'])
                lab = 'True' if p else 'Uncertain'
            y = str(r['label']).strip()
            c2 += (p == (y == 'True'))                 # binary: entailed or not
            c3 += (lab == y)                           # 3-way (False vs Uncertain separated only for gold FOL)
        report(f"FOLIO/{split}/{'goldFOL' if use_gold else 'NL->FOL'}", c2, len(df), time.time() - t0, f"3-way={c3/len(df)*100:.1f}%" if use_gold else '')


Dataset: FOLIO/train/goldFOL
Correct Predictions: 238 / 300
Accuracy: 79.33 %
Execution Time: 33.57 seconds
3-way=62.3%

Dataset: FOLIO/train/NL->FOL
Correct Predictions: 188 / 300
Accuracy: 62.67 %
Execution Time: 0.28 seconds


Dataset: FOLIO/validation/goldFOL
Correct Predictions: 165 / 203
Accuracy: 81.28 %
Execution Time: 62.28 seconds
3-way=67.0%

Dataset: FOLIO/validation/NL->FOL
Correct Predictions: 129 / 203
Accuracy: 63.55 %
Execution Time: 0.12 seconds



In [20]:
# FOL REASONING TEST

tests = [
    {
        "premises": [
            "∀x (Human(x) → Mortal(x))",
            "Human(Socrates)"
        ],
        "query": "Mortal(Socrates)",
        "expected": True
    },
    {
        "premises": [
            "∀x (Bird(x) → Animal(x))",
            "Bird(Tweety)"
        ],
        "query": "Animal(Tweety)",
        "expected": True
    },
    {
        "premises": [
            "∀x (Cat(x) → Animal(x))",
            "Cat(Tom)"
        ],
        "query": "Human(Tom)",
        "expected": False
    }
]

for i, test in enumerate(tests, 1):
    kb = FolKB(test["premises"])
    result = kb.entails(test["query"])

    print(f"Test {i}")
    print("Query:", test["query"])
    print("Expected:", test["expected"])
    print("Predicted:", result)
    print("-" * 40)

Test 1
Query: Mortal(Socrates)
Expected: True
Predicted: True
----------------------------------------
Test 2
Query: Animal(Tweety)
Expected: True
Predicted: True
----------------------------------------
Test 3
Query: Human(Tom)
Expected: False
Predicted: False
----------------------------------------


In [21]:
# Inspect negative causal effects
print("========== NEGATIVE CAUSAL EXAMPLES ==========")

count = 0

for info in cb['given_info']:
    if re.search(r'causes?\s+not\s+', info, re.I):
        eqs, obs = parse_scm(info)

        print("\nOriginal context:")
        print(info)

        print("\nParsed equations:")
        print(eqs)

        count += 1
        if count == 3:
            break


# Inspect joint causal effects
print("\n========== JOINT CAUSAL EXAMPLES ==========")

count = 0

for info in cb['given_info']:
    if re.search(
        r'\b\w+\s+and\s+\w+\s+together\s+cause',
        info,
        re.I
    ):
        eqs, obs = parse_scm(info)

        print("\nOriginal context:")
        print(info)

        print("\nParsed equations:")
        print(eqs)

        count += 1
        if count == 3:
            break

========== NEGATIVE CAUSAL EXAMPLES ==========

Original context:
We know that Ziklo causes Blaf, Blaf and Ziklo together cause Trune, Trune causes not Vork, and Vork causes Lumbo.

Parsed equations:
{'blaf': ('and', [('ziklo', False)]), 'trune': ('and', [('blaf', False), ('ziklo', False)]), 'vork': ('or', [('trune', True)]), 'lumbo': ('and', [('vork', False)])}

Original context:
We know that Glent causes Razz, Razz and Glent together cause Pex, Pex causes not Zurn, and Zurn causes Zlim.

Parsed equations:
{'razz': ('and', [('glent', False)]), 'pex': ('and', [('razz', False), ('glent', False)]), 'zurn': ('or', [('pex', True)]), 'zlim': ('and', [('zurn', False)])}

Original context:
We know that Praf causes Vank, Vank and Praf together cause Scud, Scud causes not Wrenk, and Wrenk causes Klep.

Parsed equations:
{'vank': ('and', [('praf', False)]), 'scud': ('and', [('vank', False), ('praf', False)]), 'wrenk': ('or', [('scud', True)]), 'klep': ('and', [('wrenk', False)])}

========== JOI

## 8. Generate `submission.csv`

In [22]:
# ==================== FINAL EVALUATION / SUBMISSION ====================

TEST_PATH = None
OUTPUT_PATH = "submission.csv"
MODE = "cwa"

if TEST_PATH is not None:
    if not os.path.exists(TEST_PATH):
        raise FileNotFoundError(f"Evaluation file not found: {TEST_PATH}")
    test = pd.read_csv(TEST_PATH)
    is_demo = False
else:
    # Demonstration only: this is NOT the private evaluation set.
    if "rt" not in globals() or "test" not in rt:
        raise RuntimeError(
            "No evaluation file was specified and RuleTaker was not loaded. "
            "Set TEST_PATH to the instructor-provided CSV."
        )
    test = rt["test"].to_pandas()
    is_demo = True

task_name, input_cols = infer_input_columns(test)
print("Detected input format:", task_name)
print("Detected columns:", input_cols)
print("Available columns:", test.columns.tolist())
if not input_cols:
    raise ValueError(
        "Unrecognized evaluation CSV schema. Expected one of: "
        "(context, statement), (given_info, question), "
        "(premises, conclusion), or supplied FOL columns."
    )

if "id" in test.columns:
    ids = test["id"].tolist()
elif "question_id" in test.columns:
    ids = test["question_id"].tolist()
else:
    ids = list(range(len(test)))

t0 = time.time()
predictions, failures = [], []
for idx, (_, row) in enumerate(test.iterrows()):
    try:
        predictions.append(bool(solve_record(row, mode=MODE)))
    except Exception as exc:
        predictions.append(False)  # binary output required; processing failures are separately reported
        failures.append((idx, str(exc)))

elapsed = time.time() - t0
submission = pd.DataFrame({
    "id": ids,
    "label": ["True" if p else "False" for p in predictions]
})
submission.to_csv(OUTPUT_PATH, index=False)

print("PUBLIC DEMO ONLY - replace TEST_PATH when the instructor supplies evaluation logic." if is_demo else "Instructor evaluation input processed.")
print(f"Predictions generated: {len(predictions)}")
print(f"Inference time: {elapsed:.3f} seconds")
print(f"Average time/query: {elapsed / max(len(predictions), 1):.6f} seconds")
print(f"Rows with processing errors: {len(failures)}")
if failures:
    print("First 10 errors:", failures[:10])
print("Saved:", OUTPUT_PATH)
print(submission.head(10).to_string(index=False))

Detected input format: RuleTaker / context-query
Detected columns: ['context', 'statement/query/question']
Available columns: ['context', 'statement', 'reasoning', 'depth', 'flag']
PUBLIC DEMO ONLY - replace TEST_PATH when the instructor supplies evaluation logic.
Predictions generated: 250
Inference time: 0.038 seconds
Average time/query: 0.000152 seconds
Rows with processing errors: 0
Saved: submission.csv
 id label
  0 False
  1 False
  2 False
  3 False
  4 False
  5 False
  6 False
  7 False
  8 False
  9 False


In [23]:
print("Prediction distribution:")
print(submission["label"].value_counts())

print("\nSubmission shape:")
print(submission.shape)

print("\nFirst 20 predictions:")
print(submission.head(20))

Prediction distribution:
label
True     162
False     88
Name: count, dtype: int64

Submission shape:
(250, 2)

First 20 predictions:
    id  label
0    0  False
1    1  False
2    2  False
3    3  False
4    4  False
5    5  False
6    6  False
7    7  False
8    8  False
9    9  False
10  10   True
11  11  False
12  12  False
13  13  False
14  14  False
15  15  False
16  16  False
17  17  False
18  18  False
19  19  False


In [24]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Actual labels from RuleTaker
y_true = test["flag"].apply(to_bool)

# Your predicted labels
y_pred = submission["label"].apply(to_bool)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

print("Public RuleTaker Demo Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Public RuleTaker Demo Accuracy: 97.6 %

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.94      0.97        94
        True       0.96      1.00      0.98       156

    accuracy                           0.98       250
   macro avg       0.98      0.97      0.97       250
weighted avg       0.98      0.98      0.98       250


Confusion Matrix:
[[ 88   6]
 [  0 156]]


In [25]:
import os

print("Current directory:", os.getcwd())
print("CSV exists:", os.path.exists("submission.csv"))
print("CSV size:", os.path.getsize("submission.csv"), "bytes")


Current directory: /content
CSV exists: True
CSV size: 2237 bytes


In [26]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>